In [23]:
import pandas as pd

In [24]:
df1 = pd.read_csv("data/02-Secretaria/PASAJEROS_TRANSPORTES ARANJUEZ SANTA CRUZ.csv", encoding="latin-1", sep=";")
df2 = pd.read_csv("data/02-Secretaria/RECORRIDOS_TRANSPORTES ARANJUEZ SANTA CRUZ.csv", encoding="latin-1", sep=";")

FileNotFoundError: [Errno 2] No such file or directory: 'data/02-Secretaria/PASAJEROS_TRANSPORTES ARANJUEZ SANTA CRUZ.csv'

In [ ]:
df1.merge(df2, how="inner", on="NOMBREARCHIVO")

In [ ]:
import re
import json
from pathlib import Path

def extraer_red_peatonal(html_path: Path) -> list:
    with open(html_path, "r", encoding="utf-8") as f:
        contenido = f.read()

    # Regex breakdown:
    # 1. Finds L.polyline( ... )
    # 2. Captures the content inside the first bracket: the list of coordinates [[...]]
    # 3. Ensures it contains the specific "Red Peatonal" color: #000000
    patron_completo = r"L\.polyline\(\s*(\[\[.*?\]\]).*?\"color\":\s*\"#000000\""
    
    # We use re.DOTALL so the dot matches newlines if the JS is formatted across lines
    coincidencias = re.findall(patron_completo, contenido, re.DOTALL)
    
    red_peatonal = []
    
    for coord_string in coincidencias:
        try:
            # Convert the string format [[lat, lon], ...] directly into a Python list
            coords = json.loads(coord_string)
            red_peatonal.append({
                "coords": coords,
                "color": "#000000",
                "tipo": "Red Peatonal"
            })
        except json.JSONDecodeError:
            continue
            
    return red_peatonal

In [ ]:
def extraer_infraestructura(html_path: Path, color_hex: str) -> list:
    with open(html_path, "r", encoding="utf-8") as f:
        contenido = f.read()
    
    # Dynamically look for the specific hex color
    patron = rf"L\.polyline\(\s*(\[\[.*?\]\]).*?\"color\":\s*\"{color_hex}\""
    coincidencias = re.findall(patron, contenido, re.DOTALL)
    
    segmentos = []
    for coord_string in coincidencias:
        try:
            coords = json.loads(coord_string)
            segmentos.append(coords)
        except:
            continue
    return segmentos

ciclorrutas = extraer_infraestructura("mapas/ruta_041_paraderos_aranjuez_suaves_peatonal_ciclorrutas_unificado.html", "#9370DB")
peatonal = extraer_infraestructura("mapas/ruta_041_paraderos_aranjuez_suaves_peatonal_ciclorrutas_unificado.html", "#000000")

In [ ]:
with open("data/ciclorutas_coords.json","w")as f:
    json.dump({"coords":ciclorrutas},f,indent=2)

In [ ]:
with open("data/red_peatonal_coords.json","w")as f:
    json.dump({"coords":peatonal},f,indent=2)

In [ ]:
data = extraer_red_peatonal("ruta_041_paraderos_aranjuez_suaves_peatonal_ciclorrutas_unificado.html")

In [97]:
import pandas as pd
import numpy as np
import json
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [98]:
df = pd.read_csv("data/rich_041.csv")
with open("data/rutas.json","r") as f:
    rutas = json.load(f)
ruta = rutas["041"]

In [99]:
m = folium.Map(location=ruta[0], zoom_start=13)

folium.PolyLine(ruta).add_to(m)

m

In [100]:
lat = []
lon = []

for la, lo in ruta:
    lat.append(la)
    lon.append(lo)

In [101]:
ruta_df = pd.DataFrame({"lon":lon,"lat":lat})

In [102]:
from sklearn.neighbors import BallTree
import numpy as np

# --- Build BallTree on ruta_df coordinates (haversine needs radians) ---
ruta_coords_rad = np.radians(ruta_df[["lat", "lon"]].values)
tree = BallTree(ruta_coords_rad, metric="haversine")

# --- Query: for each point in df, find distance to nearest ruta point ---
df_coords_rad = np.radians(df[["latitud", "longitud"]].values)

# distances are in radians → multiply by Earth radius (meters) to get meters
distances, indices = tree.query(df_coords_rad, k=1)

EARTH_RADIUS_M = 6_371_000  # meters
df["dist_to_ruta_m"] = distances[:, 0] * EARTH_RADIUS_M
df["nearest_ruta_idx"] = indices[:, 0]

print(f"Points before filter : {len(df)}")
print(df["dist_to_ruta_m"].describe())


Points before filter : 54004
count    54004.000000
mean        97.021045
std        148.274949
min          0.000000
25%          6.558343
50%         11.882901
75%        162.581189
max       5430.222362
Name: dist_to_ruta_m, dtype: float64


In [163]:
df[["latitud","longitud"]].values

array([[  6.247818, -75.565483],
       [  6.247818, -75.565483],
       [  6.247818, -75.565483],
       ...,
       [  6.247172, -75.566246],
       [  6.24661 , -75.566704],
       [  6.244355, -75.56971 ]])

In [103]:
THRESHOLD_M = 30

df_filtered = df[df["dist_to_ruta_m"] <= THRESHOLD_M].copy()

print(f"Points before filter : {len(df)}")
print(f"Points after  filter : {len(df_filtered)}")
print(f"Dropped              : {len(df) - len(df_filtered)}")


Points before filter : 54004
Points after  filter : 33519
Dropped              : 20485


In [104]:
THRESHOLD_M = 30

df_filtered = df[df["dist_to_ruta_m"] <= THRESHOLD_M].copy()

print(f"Points before filter : {len(df)}")
print(f"Points after  filter : {len(df_filtered)}")
print(f"Dropped              : {len(df) - len(df_filtered)}")


Points before filter : 54004
Points after  filter : 33519
Dropped              : 20485


In [105]:
df_filtered["horaregistro"] = pd.to_timedelta(df_filtered["horaregistro"])

In [ ]:
df_filtered.groupby("hora")

In [106]:
df_dia = df_filtered[(df_filtered.horaregistro.dt.components.hours.between(12,14)) & (df_filtered["fecha"] == "2026-03-19")]

In [157]:
df_filtered[df_filtered.horaregistro.dt.components.hours.between(12,14)]["fecha"].value_counts().idxmax()

'2026-03-17'

In [108]:
placas = df_dia["placavehiculo"].unique()
# 1. Generate 23 distinct hex colors using a library like 'tab20' or 'viridis'
# 'tab20' has 20 unique colors, 'nipy_spectral' is good for large sets
cmap = plt.get_cmap('nipy_spectral', len(placas)) 
colors_hex = [mcolors.to_hex(cmap(i)) for i in range(len(placas))]

# 2. Create the dictionary mapping
placa_color_map = dict(zip(placas, colors_hex))

# 3. Initialize Map
m = folium.Map(location=[df_dia.iloc[0]['latitud'], df_dia.iloc[0]['longitud']], zoom_start=13)

# 4. Add to map using CircleMarker (supports Hex)
for index, row in df_dia.iterrows():
    placa = row['placavehiculo']
    
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=8,
        color=placa_color_map[placa],  # Border color
        fill=True,
        fill_color=placa_color_map[placa], # Fill color
        fill_opacity=0.7,
        popup=f"Placa: {placa}"
    ).add_to(m)

m

In [109]:
np.radians(df_dia[["latitud", "longitud"]].values)


array([[ 0.10971407, -1.31859578],
       [ 0.10967   , -1.31859711],
       [ 0.10956945, -1.31862426],
       ...,
       [ 0.10968443, -1.3185959 ],
       [ 0.10968583, -1.31859604],
       [ 0.10971862, -1.31859377]])

In [114]:
indices[0][0]

139

In [123]:
point = [[6.274865, -75.551662]]
# --- Build BallTree on ruta_df coordinates (haversine needs radians) ---
posiciones = np.radians(df_dia[["latitud", "longitud"]].values)
tree = BallTree(posiciones, metric="haversine")

# --- Query: for each point in df, find distance to nearest ruta point ---
point = np.radians(point)

# distances are in radians → multiply by Earth radius (meters) to get meters
distances, indices = tree.query(point, k=3)

In [158]:
df_dia.iloc[indices[0]].placavehiculo.unique()

array(['FWK922', 'TSK626', 'EQS783'], dtype=object)

In [147]:
pd.to_datetime(pd.Timestamp.now()).hour

17

In [137]:
df_filtered.to_csv("data/clean_historic.csv")